In [1]:
import h5py
import numpy as np
import xarray as xr
from pyproj import Transformer
import matplotlib.pyplot as plt

In [5]:
h5_path = "/mnt/storage/brencher/repos/jirp_nisar_eval/data/nisar_GLSC/NISAR_L2_PR_GSLC_005_135_A_031_4005_DHDH_A_20251119T132333_20251119T132408_X05009_N_F_J_001.h5"
bbox = (-134.66, 58.71, -133.96, 58.91)  # lon_min, lat_min, lon_max, lat_max

freq = "frequencyA"
pol = "HH"   # change to HV, VV, etc. if needed

In [3]:
def find_first_key(group, candidates):
    """Find the first candidate key present in an HDF5 group."""
    for key in candidates:
        if key in group:
            return key
    raise KeyError(f"None of these keys found: {candidates}. Available keys: {list(group.keys())}")


def find_epsg(h5):
    """
    Try common places/attributes for EPSG.
    NISAR products usually store projection info, but exact naming can vary
    between product versions/sample products.
    """
    # First try common dataset names by walking the file
    epsg_hits = []

    def visitor(name, obj):
        lname = name.lower()
        if isinstance(obj, h5py.Dataset) and ("epsg" in lname or "projection" in lname):
            epsg_hits.append(name)

    h5.visititems(visitor)

    for name in epsg_hits:
        try:
            val = h5[name][()]
            if np.size(val) == 1:
                val = int(np.asarray(val).item())
                if 1000 < val < 100000:
                    return val
        except Exception:
            pass

    # Then try attributes
    attr_hits = []

    def attr_visitor(name, obj):
        for k, v in obj.attrs.items():
            if "epsg" in k.lower():
                attr_hits.append((name, k, v))

    h5.visititems(attr_visitor)

    for name, k, v in attr_hits:
        try:
            val = int(np.asarray(v).item())
            if 1000 < val < 100000:
                return val
        except Exception:
            pass

    raise ValueError(
        "Could not automatically find EPSG code. "
        "Inspect the file for projection/epsg metadata, or manually set epsg=xxxx."
    )


def crop_slice_from_coords(coord, vmin, vmax):
    """
    Return slice for coordinate vector, handling ascending or descending coords.
    """
    lo, hi = min(vmin, vmax), max(vmin, vmax)
    mask = (coord >= lo) & (coord <= hi)

    if not np.any(mask):
        raise ValueError(
            f"Bbox does not overlap coordinate range. "
            f"Requested {lo:.2f} to {hi:.2f}; coord range is {coord.min():.2f} to {coord.max():.2f}"
        )

    idx = np.where(mask)[0]
    return slice(idx.min(), idx.max() + 1)

In [6]:
with h5py.File(h5_path, "r") as f:
    group_path = f"/science/LSAR/GSLC/grids/{freq}"
    g = f[group_path]

    print("Available datasets in GSLC frequency group:")
    print(list(g.keys()))

    # Common coordinate names; adjust if your file uses different names.
    x_key = find_first_key(g, ["xCoordinates", "xCoordinatesSpacing", "x", "longitude"])
    y_key = find_first_key(g, ["yCoordinates", "yCoordinatesSpacing", "y", "latitude"])

    # Most products should use xCoordinates/yCoordinates.
    # If the helper accidentally finds spacing variables instead, inspect list(g.keys()) manually.
    if "Spacing" in x_key or "Spacing" in y_key:
        raise ValueError(
            f"Found {x_key=} and {y_key=}, which look like spacing fields, not coordinate vectors. "
            "Print list(g.keys()) and choose the true coordinate vector names."
        )

    x = g[x_key][:]
    y = g[y_key][:]

    epsg = find_epsg(f)
    print(f"Using EPSG:{epsg}")

    # Transform lon/lat bbox corners to the GSLC projected CRS
    lon_min, lat_min, lon_max, lat_max = bbox
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)

    corners_lon = [lon_min, lon_min, lon_max, lon_max]
    corners_lat = [lat_min, lat_max, lat_min, lat_max]
    corners_x, corners_y = transformer.transform(corners_lon, corners_lat)

    x_slice = crop_slice_from_coords(x, min(corners_x), max(corners_x))
    y_slice = crop_slice_from_coords(y, min(corners_y), max(corners_y))

    z = g[pol][y_slice, x_slice]
    x_crop = x[x_slice]
    y_crop = y[y_slice]

Available datasets in GSLC frequency group:
['HH', 'HV', 'azimuthBandwidth', 'centerFrequency', 'listOfPolarizations', 'mask', 'numberOfSubSwaths', 'projection', 'rangeBandwidth', 'slantRangeSpacing', 'xCoordinateSpacing', 'xCoordinates', 'yCoordinateSpacing', 'yCoordinates', 'zeroDopplerTimeSpacing']
Using EPSG:32608


In [7]:
# Put complex GSLC crop into xarray
da_slc = xr.DataArray(
    z,
    dims=("y", "x"),
    coords={"y": y_crop, "x": x_crop},
    name=f"{pol}_complex",
    attrs={
        "source": "NISAR GSLC",
        "polarization": pol,
        "epsg": epsg,
        "crs": f"EPSG:{epsg}",
    },
)

# Convert complex GSLC to intensity/power
da_intensity = np.abs(da_slc) ** 2
da_intensity.name = f"{pol}_intensity"

# Multilook intensity to reduce speckle.
# Use larger factors for a smoother/more GRD-like image.
look_y = 4
look_x = 4

da_intensity_ml = (
    da_intensity
    .coarsen(y=look_y, x=look_x, boundary="trim")
    .mean()
)
da_intensity_ml.name = f"{pol}_intensity_multilooked"

# Convert to dB for display.
# Add a tiny floor to avoid log10(0).
eps = np.finfo("float32").tiny
da_db = 10 * np.log10(da_intensity_ml.where(da_intensity_ml > 0) + eps)
da_db.name = f"{pol}_intensity_multilooked_db"

ds = xr.Dataset(
    {
        da_slc.name: da_slc,
        da_intensity.name: da_intensity,
        da_intensity_ml.name: da_intensity_ml,
        da_db.name: da_db,
    }
)

ds

<xarray.Dataset> Size: 1GB
Dimensions:                      (y: 5671, x: 10166)
Coordinates:
  * y                            (y) float64 45kB 6.508e+06 ... 6.53e+06
  * x                            (x) float64 81kB 5.196e+05 ... 5.602e+05
Data variables:
    HH_complex                   (y, x) complex64 461MB (-0.36083984-0.038696...
    HH_intensity                 (y, x) float32 231MB 0.1317 ... 0.005105
    HH_intensity_multilooked     (y, x) float32 231MB nan nan nan ... nan nan
    HH_intensity_multilooked_db  (y, x) float32 231MB nan nan nan ... nan nan

In [8]:
import numpy as np
import rasterio
from rasterio.transform import from_origin

def write_xarray_geotiff(da, out_tif, epsg, nodata=-9999.0, dtype="float32"):
    """
    Write a 2D xarray DataArray with dims ('y', 'x') and projected x/y coords to GeoTIFF.
    Assumes x and y are regularly spaced projected coordinates.
    """

    da = da.transpose("y", "x")

    x = da["x"].values
    y = da["y"].values
    arr = da.values.astype(dtype)

    # Replace NaNs with nodata
    arr = np.where(np.isfinite(arr), arr, nodata).astype(dtype)

    # Pixel spacing
    dx = float(np.abs(np.nanmedian(np.diff(x))))
    dy = float(np.abs(np.nanmedian(np.diff(y))))

    # GeoTIFF transform expects upper-left outer corner.
    # x/y coords are usually pixel centers.
    west = float(x.min() - dx / 2)
    north = float(y.max() + dy / 2)

    transform = from_origin(west, north, dx, dy)

    # Rasterio expects row 0 to be the northern/top row.
    # If y is ascending south-to-north, flip array vertically.
    if y[0] < y[-1]:
        arr = arr[::-1, :]

    with rasterio.open(
        out_tif,
        "w",
        driver="GTiff",
        height=arr.shape[0],
        width=arr.shape[1],
        count=1,
        dtype=dtype,
        crs=f"EPSG:{epsg}",
        transform=transform,
        nodata=nodata,
        compress="deflate",
        tiled=True,
        blockxsize=256,
        blockysize=256,
        BIGTIFF="IF_SAFER",
    ) as dst:
        dst.write(arr, 1)

    print(f"Wrote {out_tif}")

In [9]:
write_xarray_geotiff(
    da_intensity_ml,
    "nisar_gslc_intensity_multilooked_linear.tif",
    epsg=epsg,
)

Wrote nisar_gslc_intensity_multilooked_linear.tif
